# Object Detection — Exercises

Companion to the note [Object Detection](Object%20Detection.md).

Practise IoU, box-regression targets, focal loss, average precision and the anchor/imbalance arithmetic by hand, then implement vectorised IoU, box encoding/decoding, non-maximum suppression, focal loss, detection-to-ground-truth matching, average precision and Hungarian matching (as in DETR) from scratch in NumPy and SciPy.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.

In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as sp_softmax
from scipy.optimize import linear_sum_assignment
from itertools import permutations
rng = np.random.default_rng(5)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Backbone, neck, head
*🧠 Concept · ★☆☆*

Name the three parts shared by most detectors and what each does. Why do detectors use a Feature Pyramid Network (FPN) in the neck, and which kinds of objects benefit?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of how large and small objects appear in feature maps of different depth and resolution.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Backbone** (ResNet, EfficientNet, ViT): image → feature maps. **Neck** (FPN): merges fine (high-resolution, shallow) and coarse (low-resolution, semantically strong) feature maps. **Head**: outputs a class score and box for each location/query.
Objects come in many sizes; deep maps have strong semantics but low resolution (small objects vanish), shallow maps have fine detail but weak semantics. An FPN gives *every* scale both, which helps especially **small objects**.

</details>

### Exercise 2 · Two-stage, one-stage and DETR
*🧠 Concept · ★★☆*

Describe in two sentences each how Faster R-CNN, YOLO/RetinaNet and DETR produce their boxes. For each: how are duplicate boxes avoided, and what is the main trade-off?
Which would you pick for a real-time camera on an edge device, and which for the best offline accuracy on small objects?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the table of detector families: proposals, dense anchors, object queries.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Faster R-CNN**: an RPN proposes ~2,000 regions, RoI pooling crops features, a head classifies and refines each. Duplicates removed by **NMS**. Accurate, slower.
- **YOLO / RetinaNet**: predict class + box offsets for every anchor/grid cell in one pass, then NMS. Fast; needs a fix for background imbalance (focal loss in RetinaNet).
- **DETR**: a fixed set of object queries is matched to ground truth with the **Hungarian algorithm**; no anchors, no NMS (set prediction). Elegant, originally slow to train (fixed by Deformable/RT-DETR).

Real-time on the edge: a one-stage (YOLO-style) detector. Best offline accuracy on small objects: a two-stage or high-resolution FPN-based detector (also tiling), measured with latency *and* mAP on the target hardware.

</details>

### Exercise 3 · Metrics: why not accuracy, and what mAP means
*🧠 Concept · ★★☆*

1. Why is classification accuracy meaningless for detection?
2. What is the difference between mAP@0.5 and COCO mAP@[.5:.95]? Why can a detector have high mAP@0.5 but low mAP@[.5:.95]?
3. Is NMS part of training? How do DETR-style models avoid it?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

There is no fixed number of predictions per image, and a prediction can be right about the class but wrong about where.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. The number of objects is unknown; predictions can be missing (false negatives), duplicated or spurious (false positives), and a correct class with a bad box is still wrong. We need precision, recall and AP.
2. mAP@0.5 counts a prediction as TP if IoU $\ge0.5$ (lenient); COCO averages AP over IoU $0.50,0.55,\dots,0.95$, so tight boxes are rewarded. Loose-but-roughly-right boxes give a high mAP@0.5 and a low mAP@[.5:.95].
3. NMS is post-processing at inference (training uses matching/assignment instead). DETR uses one-to-one Hungarian matching, so each object is predicted exactly once and duplicates are already penalised in training.

</details>

## Part B · By hand

### Exercise 4 · IoU by hand
*✍️ By hand · ★☆☆*

Boxes are $(x_1,y_1,x_2,y_2)$. Compute the IoU of $A=(1,1,5,4)$ and $B=(3,2,7,6)$: the intersection area, the union area and the IoU. Would the prediction count as a true positive at threshold 0.5?

In [ ]:
inter = None
union = None
iou_hand = None
check('intersection', inter, 4); check('union', union, 24); check('IoU', iou_hand, 1 / 6, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Intersection corners: $(\max x_1,\max y_1,\min x_2,\min y_2)$. Union $=|A|+|B|-|A\cap B|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Intersection $=(3,2,5,4)$: area $2\cdot2=4$. $|A|=4\cdot3=12$, $|B|=4\cdot4=16$, union $=12+16-4=24$. IoU $=4/24=1/6\approx0.167$: a **false positive** at 0.5, even though the boxes overlap.

```python
inter = 4
union = 24
iou_hand = 4 / 24
```

</details>

### Exercise 5 · Box regression targets
*✍️ By hand · ★★☆*

An anchor in $(c_x,c_y,w,h)$ format is $a=(50,50,40,80)$ and the ground-truth box is $(60,40,80,40)$. Compute the regression targets $t_x=\frac{x-a_x}{a_w}$, $t_y=\frac{y-a_y}{a_h}$, $t_w=\log\frac{w}{a_w}$, $t_h=\log\frac{h}{a_h}$.
Why does the detector predict *offsets relative to an anchor* with a log for the size instead of raw coordinates?

In [ ]:
t_hand = None   # [tx, ty, tw, th]
check('targets', t_hand, [0.25, -0.125, np.log(2), -np.log(2)], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$t_w=\log(80/40)$, $t_h=\log(40/80)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$t_x=(60-50)/40=0.25$, $t_y=(40-50)/80=-0.125$, $t_w=\ln2=0.693$, $t_h=\ln0.5=-0.693$.
Offsets are **scale-invariant** (a small error on a big box and a small box give comparable targets) and roughly centred around 0, which makes regression easy; the log makes width/height changes multiplicative and symmetric (doubling and halving are $\pm\ln2$) and guarantees positive sizes when decoding with $\exp$.

```python
t_hand = [0.25, -0.125, np.log(2), np.log(0.5)]
```

</details>

### Exercise 6 · Focal loss numbers
*✍️ By hand · ★★☆*

With $\mathrm{FL}(p_t)=-(1-p_t)^\gamma\log p_t$ ($\alpha_t=1$, $\gamma=2$) compute the focal loss and the cross-entropy $-\log p_t$ for $p_t\in\{0.9,\,0.5,\,0.1\}$ (natural log, 4 decimals).
By what factor is the easy example ($p_t=0.9$) down-weighted compared with plain cross-entropy?

In [ ]:
ce_hand = None   # for p_t = 0.9, 0.5, 0.1
fl_hand = None
down = None      # down-weighting factor of the easy example
_pt = np.array([0.9, 0.5, 0.1])
check('cross-entropy', ce_hand, -np.log(_pt), tol=1e-3); check('focal loss', fl_hand, -(1 - _pt) ** 2 * np.log(_pt), tol=1e-3)
check('down-weighting of the easy example', down, 0.01, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

$-\ln0.9=0.1054$, $-\ln0.5=0.6931$, $-\ln0.1=2.3026$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| $p_t$ | CE | $(1-p_t)^2$ | FL |
|---|---|---|---|
| 0.9 | 0.1054 | 0.01 | 0.0011 |
| 0.5 | 0.6931 | 0.25 | 0.1733 |
| 0.1 | 2.3026 | 0.81 | 1.8651 |

The easy example is down-weighted by $(1-0.9)^2=0.01$, i.e. $100\times$, the hard example barely changes. The thousands of easy background anchors no longer dominate the loss.

```python
ce_hand = [0.1054, 0.6931, 2.3026]
fl_hand = [0.0011, 0.1733, 1.8651]
down = 0.01
```

</details>

### Exercise 7 · Average precision by hand
*✍️ By hand · ★★☆*

For one class there are 4 ground-truth objects. The detector outputs 5 boxes, sorted by confidence, with results TP, FP, TP, FP, TP (the fourth object is missed).
Write down precision and recall at each rank, then compute the AP as the area under the interpolated precision–recall curve (precision replaced by the maximum precision at any recall $\ge r$).

In [ ]:
ap_hand = None
check('AP', ap_hand, 0.5667, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Precision at rank $k$ = TPs so far / $k$; recall = TPs so far / 4. Precision envelope from the right: $1,\ 0.667,\ 0.6$ at recalls $0.25,\ 0.5,\ 0.75$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| rank | result | precision | recall |
|---|---|---|---|
| 1 | TP | 1.000 | 0.25 |
| 2 | FP | 0.500 | 0.25 |
| 3 | TP | 0.667 | 0.50 |
| 4 | FP | 0.500 | 0.50 |
| 5 | TP | 0.600 | 0.75 |

Interpolated precision at recall levels $0.25,0.5,0.75$: $1,\ 0.667,\ 0.6$ (maximum to the right). $\mathrm{AP}=0.25\cdot1+0.25\cdot0.667+0.25\cdot0.6=0.567$ (nothing is gained beyond recall 0.75).

```python
ap_hand = 0.25 * 1 + 0.25 * 2 / 3 + 0.25 * 0.6
```

</details>

### Exercise 8 · How many predictions? The background imbalance
*✍️ By hand · ★☆☆*

A YOLO-style detector uses a $640\times640$ input, three feature levels with strides 8, 16 and 32, and 3 anchors per cell; each prediction has $4+1+80=85$ numbers.
How many predictions per image (`n_preds`) and how many output numbers (`n_outputs`)? If the image has 10 objects and each is matched to 3 anchors, what is the background : foreground ratio (`bg_fg`)? What does it motivate?

In [ ]:
n_preds = None
n_outputs = None
bg_fg = None
check('predictions', n_preds, 25200); check('output numbers', n_outputs, 2142000); check('background : foreground', bg_fg, 839, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Cells per level: $(640/8)^2+(640/16)^2+(640/32)^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Cells: $80^2+40^2+20^2=6400+1600+400=8400$; times 3 anchors $=25{,}200$ predictions and $25{,}200\cdot85=2{,}142{,}000$ numbers. Foreground $=30$, background $=25{,}170$, ratio $\approx839:1$.
This extreme imbalance is why one-stage detectors need focal loss (RetinaNet), hard-negative mining (SSD) or an assignment scheme with a separate objectness score (YOLO).

```python
n_preds = (80**2 + 40**2 + 20**2) * 3
n_outputs = n_preds * 85
bg_fg = (n_preds - 30) / 30
```

</details>

### Exercise 9 · COCO mAP for one object
*✍️ By hand · ★☆☆*

An image has a single object and the detector predicts one box of the right class with IoU $=0.78$. For each threshold in $\{0.50,0.55,\dots,0.95\}$ the AP is 1 if IoU $\ge$ threshold, else 0.
At how many thresholds is it a true positive (`n_tp`)? What are mAP@0.5 (`map50`) and the COCO mAP@[.5:.95] (`map_coco`) for this image?

In [ ]:
n_tp = None
map50 = None
map_coco = None
_thr = np.arange(0.5, 0.96, 0.05)
check('thresholds met', n_tp, int((0.78 >= _thr - 1e-9).sum())); check('mAP@0.5', map50, 1.0); check('COCO mAP', map_coco, np.mean(0.78 >= _thr - 1e-9))

<details>
<summary><b>💡 Hint</b></summary>

The thresholds $0.50,\dots,0.75$ are all $\le0.78$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Thresholds $0.50,0.55,0.60,0.65,0.70,0.75$ are met ($0.78\ge$ each), $0.80,\dots,0.95$ are not: 6 of 10. So mAP@0.5 $=1.0$ but COCO mAP $=0.6$: tight localisation matters.

```python
n_tp = 6
map50 = 1.0
map_coco = 0.6
```

</details>

## Part C · Code from scratch

### Exercise 10 · Vectorised IoU
*💻 Code · ★☆☆*

Implement `iou(a, b)` for arrays of boxes in $(x_1,y_1,x_2,y_2)$ format, `a` of shape `(N, 4)` and `b` of shape `(M, 4)`, returning the $N\times M$ matrix of IoUs without Python loops.
Test: the example of Exercise 4, symmetry, $\mathrm{IoU}(A,A)=1$, disjoint and edge-touching boxes (0), and containment.

In [ ]:
def iou(a, b):
    return None
A = np.array([[1, 1, 5, 4], [0, 0, 10, 10.]]); Bx = np.array([[3, 2, 7, 6], [0, 0, 10, 10], [20, 20, 30, 30], [10, 0, 20, 10], [2, 2, 4, 4.]])
M = iou(A, Bx)
check('example of Exercise 4', None if M is None else M[0, 0], 1 / 6)
check('IoU(A, A) = 1', None if M is None else M[1, 1], 1.0)
check('disjoint boxes: 0', None if M is None else M[1, 2], 0.0); check('edge-touching boxes: 0', None if M is None else M[1, 3], 0.0)
check('containment: 4/100', None if M is None else M[1, 4], 0.04)
check('symmetry', None if M is None else iou(Bx, A), None if M is None else M.T)

<details>
<summary><b>💡 Hint</b></summary>

Use broadcasting: `a[:, None, :2]` against `b[None, :, :2]` with `np.maximum` for the top-left corner and `np.minimum` for the bottom-right corner. Clip the width/height of the intersection at 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Intersection corners are the max of the top-left corners and the min of the bottom-right corners; width and height are clipped at 0 so that disjoint boxes give 0. The union is the sum of both areas minus the intersection.
This $N\times M$ matrix is the building block of NMS, label assignment, matching and AP.

```python
def iou(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)
```

</details>

### Exercise 11 · Encoding and decoding box offsets
*💻 Code · ★★☆*

Implement `xyxy_to_cxcywh(b)`, `encode(gt, anchor)` and `decode(t, anchor)` for arrays of shape `(N, 4)` where `gt` and `anchor` are in $(c_x,c_y,w,h)$ format and the targets are $(t_x,t_y,t_w,t_h)$ as in Exercise 5.
Check Exercise 5, the round trip `decode(encode(gt, a), a) == gt`, and that the encoded size targets of a prediction identical to its anchor are exactly 0.

In [ ]:
def xyxy_to_cxcywh(b):
    return None

def encode(gt, anchor):
    return None

def decode(t, anchor):
    return None
a_ = np.array([[50, 50, 40, 80.]]); g_ = np.array([[60, 40, 80, 40.]])
check('Exercise 5 targets', encode(g_, a_), [[0.25, -0.125, np.log(2), -np.log(2)]])
rg, ra = np.c_[rng.uniform(10, 90, (6, 2)), rng.uniform(5, 50, (6, 2))], np.c_[rng.uniform(10, 90, (6, 2)), rng.uniform(5, 50, (6, 2))]
check('round trip decode(encode(gt))', None if encode(rg, ra) is None else decode(encode(rg, ra), ra), rg)
check('box = anchor -> targets 0', encode(ra, ra), np.zeros((6, 4)))
check('xyxy -> cxcywh', xyxy_to_cxcywh(np.array([[0, 0, 4, 4.], [2, 2, 6, 10]])), [[2, 2, 4, 4], [4, 6, 4, 8]])

<details>
<summary><b>💡 Hint</b></summary>

`encode`: `(gt[:, :2] - a[:, :2]) / a[:, 2:]` and `np.log(gt[:, 2:] / a[:, 2:])`. `decode` inverts both.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`decode` applies the inverse: centre $=a_c+t_{xy}\,a_{wh}$, size $=a_{wh}\,e^{t_{wh}}$. A box equal to its anchor gives all-zero targets, so the network can start from "predict the anchor" and only learn corrections.

```python
def xyxy_to_cxcywh(b):
    return np.c_[(b[:, :2] + b[:, 2:]) / 2, b[:, 2:] - b[:, :2]]

def encode(gt, anchor):
    return np.c_[(gt[:, :2] - anchor[:, :2]) / anchor[:, 2:], np.log(gt[:, 2:] / anchor[:, 2:])]

def decode(t, anchor):
    return np.c_[anchor[:, :2] + t[:, :2] * anchor[:, 2:], anchor[:, 2:] * np.exp(t[:, 2:])]
```

</details>

### Exercise 12 · Non-maximum suppression
*💻 Code · ★★☆*

Implement greedy `nms(boxes, scores, thr)` returning the **indices** of the kept boxes in order of decreasing score: repeatedly keep the highest-scoring remaining box and discard all remaining boxes whose IoU with it is **greater** than `thr`.
Test it on a hand-made case and with two properties on random boxes: no two kept boxes overlap more than `thr`, and every discarded box overlaps some kept box (with higher score) by more than `thr`.

In [ ]:
def nms(boxes, scores, thr):
    return None
def _iou(a, b):
    lt, rb = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    ar = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter)
bx = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [5, 5, 15, 15.]]); sc = np.array([0.9, 0.8, 0.7, 0.6])
check('thr = 0.5: keep A, C, D', nms(bx, sc, 0.5), [0, 2, 3]); check('thr = 0.1: keep A, C', nms(bx, sc, 0.1), [0, 2]); check('thr = 0.9: keep everything', nms(bx, sc, 0.9), [0, 1, 2, 3])
rb = np.c_[rng.uniform(0, 60, (40, 2)), rng.uniform(10, 40, (40, 2))]; rb[:, 2:] += rb[:, :2]; rs = rng.uniform(size=40)
kept = nms(rb, rs, 0.4)
if kept is not None:
    M = _iou(rb, rb); rest = [i for i in range(40) if i not in kept]
    check('kept boxes overlap at most thr', max([M[i, j] for i in kept for j in kept if i != j] or [0]) <= 0.4, True)
    check('each discarded box overlaps a better kept box', all(any(M[i, j] > 0.4 and rs[j] > rs[i] for j in kept) for i in rest), True)
else:
    check('properties of NMS', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Sort with `np.argsort(-scores)`. Compute the IoU of the current best box with all remaining ones (reuse `iou`, or the helper defined in the test cell) and filter.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The hand-made case: A $(0,0,10,10)$ 0.9, B $(1,1,11,11)$ 0.8, C $(20,20,30,30)$ 0.7, D $(5,5,15,15)$ 0.6. IoU(A,B)$=0.68$, IoU(A,D)$=0.14$, IoU(B,D)$=0.22$. With `thr=0.5` B is suppressed by A, D survives (A, C, D kept); with `thr=0.1` D is also suppressed by A; with `thr=0.9` nothing is removed.
NMS is a greedy heuristic; boxes of different classes are usually suppressed separately (loop over classes). Choosing `thr` trades duplicates against missing overlapping objects (crowds), which is a weakness DETR-style set prediction avoids.

```python
def nms(boxes, scores, thr):
    order = list(np.argsort(-scores))
    keep = []
    while order:
        i = order.pop(0)
        keep.append(int(i))
        if order:
            ious = _iou(boxes[i:i+1], boxes[order])[0]
            order = [j for j, v in zip(order, ious) if v <= thr]
    return keep
```

</details>

### Exercise 13 · Focal loss
*💻 Code · ★★☆*

Implement `focal_loss(p, y, alpha=0.25, gamma=2.0)` for predicted probabilities `p` of the positive class and labels `y` in $\{0,1\}$ (elementwise):
$p_t=p$ if $y=1$ else $1-p$, $\alpha_t=\alpha$ if $y=1$ else $1-\alpha$, $\mathrm{FL}=-\alpha_t(1-p_t)^\gamma\log p_t$.
Check the limit $\gamma=0,\alpha=0.5$ (half of cross-entropy). Then simulate 10,000 easy negatives ($p=0.01$) and 10 hard positives ($p=0.3$): store the share of the total loss that comes from the negatives, with plain cross-entropy (`neg_share_ce`) and with focal loss (`neg_share_fl`).

In [ ]:
def focal_loss(p, y, alpha=0.25, gamma=2.0):
    return None

neg_share_ce = None
neg_share_fl = None
pp, yy = rng.uniform(0.01, 0.99, 20), rng.integers(0, 2, 20)
_pt = np.where(yy == 1, pp, 1 - pp)
check('gamma = 0, alpha = 0.5 -> half the cross-entropy', None if focal_loss(pp, yy, 0.5, 0) is None else 2 * focal_loss(pp, yy, 0.5, 0), -np.log(_pt))
check('p_t = 0.9 is down-weighted by 100', None if focal_loss(np.array([0.9]), np.array([1]), 1.0, 2.0) is None else focal_loss(np.array([0.9]), np.array([1]), 1.0, 2.0) / (-np.log(0.9)), 0.01)
check('negatives dominate cross-entropy (> 80%)', None if neg_share_ce is None else neg_share_ce > 0.8, True)
check('negatives are negligible with focal loss (< 5%)', None if neg_share_fl is None else neg_share_fl < 0.05, True)

<details>
<summary><b>💡 Hint</b></summary>

Clip `p` to $[10^{-12},1-10^{-12}]$ before the log. The cross-entropy case is `focal_loss(p, y, alpha=0.5, gamma=0)` times 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With cross-entropy the 10,000 easy negatives contribute $10{,}000\cdot0.01005\approx100.5$ against $10\cdot1.204=12.0$ for the positives: $89\%$ of the loss is "background that is already right".
With focal loss the negatives contribute $\approx0.0075$ against $\approx1.5$ for the positives: $0.5\%$. The gradient now comes from the hard examples.

```python
def focal_loss(p, y, alpha=0.25, gamma=2.0):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    pt = np.where(y == 1, p, 1 - p)
    at = np.where(y == 1, alpha, 1 - alpha)
    return -at * (1 - pt) ** gamma * np.log(pt)

p_sim = np.r_[np.full(10000, 0.01), np.full(10, 0.3)]
y_sim = np.r_[np.zeros(10000), np.ones(10)]
ce = focal_loss(p_sim, y_sim, alpha=0.5, gamma=0) * 2
fl = focal_loss(p_sim, y_sim)
neg_share_ce = ce[:10000].sum() / ce.sum()
neg_share_fl = fl[:10000].sum() / fl.sum()
```

</details>

### Exercise 14 · Matching detections to ground truth
*💻 Code · ★★☆*

Implement `match_detections(pred_boxes, pred_scores, gt_boxes, thr=0.5)` returning a boolean array `tp` in order of **decreasing score**: go through the predictions from the most to the least confident, find the ground-truth box with the highest IoU, and mark the prediction as TP if that IoU $\ge$ `thr` **and** the box is not yet matched; otherwise it is a FP (so duplicates are false positives).

In [ ]:
def match_detections(pred_boxes, pred_scores, gt_boxes, thr=0.5):
    return None
gt_s = np.array([[0, 0, 10, 10], [20, 20, 30, 30], [50, 50, 60, 60.]])
pr_s = np.array([[1, 1, 10, 10], [0, 0, 10, 10.5], [21, 21, 31, 31], [40, 40, 45, 45], [50, 50, 56, 56.]]); sc_s = np.array([0.95, 0.9, 0.8, 0.7, 0.6])
tp_s = match_detections(pr_s, sc_s, gt_s)
check('TP / FP flags', tp_s, [True, False, True, False, False])
perm = [3, 0, 4, 2, 1]
check('order of the input does not matter', match_detections(pr_s[perm], sc_s[perm], gt_s), [True, False, True, False, False])

<details>
<summary><b>💡 Hint</b></summary>

Keep a boolean array `matched` over the ground truths. Use `iou` (or `_iou` from the NMS cell) for the $N\times M$ matrix.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is exactly the bookkeeping behind Pascal VOC / COCO evaluation. In the test scene: the most confident box matches object 0; the second box also covers object 0 and is a **duplicate** (FP); the third matches object 1; the fourth is at the wrong place and the
fifth has IoU $0.36<0.5$ (FP). Object 2 is missed (a false negative, which lowers recall).

```python
def match_detections(pred_boxes, pred_scores, gt_boxes, thr=0.5):
    order = np.argsort(-pred_scores)
    M = _iou(pred_boxes[order], gt_boxes)
    matched = np.zeros(len(gt_boxes), bool)
    tp = np.zeros(len(order), bool)
    for r in range(len(order)):
        j = M[r].argmax()
        if M[r, j] >= thr and not matched[j]:
            tp[r], matched[j] = True, True
    return tp
```

</details>

### Exercise 15 · Average precision
*💻 Code · ★★☆*

Implement `average_precision(tp, n_gt)`: `tp` is a boolean array of TP/FP flags **sorted by decreasing confidence** and `n_gt` is the number of ground-truth objects. Compute precision and recall at each rank, make the precision monotonically decreasing (maximum to the right), and sum the area under the curve $\sum_k(r_k-r_{k-1})\,p^{\mathrm{interp}}_k$.
Check Exercise 7, a perfect detector, a detector with no true positives and the scene of Exercise 14.

In [ ]:
def average_precision(tp, n_gt):
    return None
check('Exercise 7: AP = 0.567', average_precision([1, 0, 1, 0, 1], 4), 0.5667, tol=1e-3)
check('perfect detector: AP = 1', average_precision([1, 1, 1], 3), 1.0)
check('no true positives: AP = 0', average_precision([0, 0, 0], 3), 0.0)
check('scene of Exercise 14: AP = 0.556', average_precision([1, 0, 1, 0, 0], 3), 5 / 9, tol=1e-3)
check('a late false positive costs less than an early one', None if average_precision([1, 1, 0], 2) is None else average_precision([1, 1, 0], 2) > average_precision([0, 1, 1], 2), True)

<details>
<summary><b>💡 Hint</b></summary>

`prec = cumsum(tp) / arange(1, K+1)`, `rec = cumsum(tp) / n_gt`. Prepend recall 0, then `np.maximum.accumulate(prec[::-1])[::-1]` gives the envelope.

</details>

<details>
<summary><b>✅ Solution</b></summary>

AP summarises the whole precision–recall trade-off in one number; ranking quality matters: a high-confidence false positive hurts more than a low-confidence one. mAP averages AP over classes, COCO mAP additionally over IoU thresholds.
For the scene of Exercise 14: tp $=[1,0,1,0,0]$, 3 objects: recalls $\tfrac13,\tfrac13,\tfrac23$, interpolated precisions $1,\ \tfrac23$ → $\mathrm{AP}=\tfrac13\cdot1+\tfrac13\cdot\tfrac23=0.556$.

```python
def average_precision(tp, n_gt):
    tp = np.asarray(tp, dtype=float)
    prec = np.cumsum(tp) / np.arange(1, len(tp) + 1)
    rec = np.cumsum(tp) / n_gt
    prec_i = np.maximum.accumulate(prec[::-1])[::-1]
    return float(np.sum(np.diff(np.r_[0, rec]) * prec_i))
```

</details>

### Exercise 16 · Hungarian matching (DETR)
*💻 Code · ★★★*

DETR assigns each ground-truth object to **exactly one** of its predictions by solving a bipartite matching. Implement `hungarian_match(iou_matrix)` for an $N_{pred}\times N_{gt}$ matrix ($N_{pred}\ge N_{gt}$) returning `(pred_idx, gt_idx)` that **maximises** the total IoU, using `scipy.optimize.linear_sum_assignment` (which minimises a cost).
Verify against brute force over all assignments, and show that greedy matching can be worse: for `[[0.9, 0.8], [0.85, 0.1]]` greedy takes (0,0) first.

In [ ]:
def hungarian_match(iou_matrix):
    return None
iou_m = rng.uniform(size=(5, 3))
res = hungarian_match(iou_m)
_best = max(sum(iou_m[p[k], k] for k in range(3)) for p in permutations(range(5), 3))
check('maximum total IoU equals brute force', None if res is None else iou_m[res[0], res[1]].sum(), _best)
check('each ground truth matched exactly once', None if res is None else sorted(res[1]), [0, 1, 2])
check('each prediction used at most once', None if res is None else len(set(res[0])) == 3, True)
M2 = np.array([[0.9, 0.8], [0.85, 0.1]]); r2 = hungarian_match(M2)
check('beats greedy (1.65 vs 1.0)', None if r2 is None else M2[r2[0], r2[1]].sum(), 1.65)

<details>
<summary><b>💡 Hint</b></summary>

Use the cost matrix `-iou_matrix`. `linear_sum_assignment(cost)` returns `(row_ind, col_ind)`. Unmatched predictions are trained to predict "no object".

</details>

<details>
<summary><b>✅ Solution</b></summary>

Greedy picks the best pair $(0,0)=0.9$ and is left with $(1,1)=0.1$, total $1.0$; the optimal assignment is $(0,1)+(1,0)=0.8+0.85=1.65$. The Hungarian algorithm finds the global optimum in $O(n^3)$.
Because matching is one-to-one, a second prediction of the same object is *not* matched and is trained to predict "no object", so duplicates are penalised during training and no NMS is needed at inference.

```python
def hungarian_match(iou_matrix):
    return linear_sum_assignment(-iou_matrix)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Object Detection](Object%20Detection.md).*